# 1. 90일치 기상 센서 로그를 직접 생성해 결측값·이상치를 보정하고 월별 리포트 만들기

In [138]:
import pandas as pd
import numpy as np

In [139]:
#1. 랜덤 시계열 Series 생성
np.random.seed(2024)

date_idx = pd.date_range(start = "1/3/2024", periods=90)
print("날짜 인덱스:\n", date_idx)

날짜 인덱스:
 DatetimeIndex(['2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06',
               '2024-01-07', '2024-01-08', '2024-01-09', '2024-01-10',
               '2024-01-11', '2024-01-12', '2024-01-13', '2024-01-14',
               '2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18',
               '2024-01-19', '2024-01-20', '2024-01-21', '2024-01-22',
               '2024-01-23', '2024-01-24', '2024-01-25', '2024-01-26',
               '2024-01-27', '2024-01-28', '2024-01-29', '2024-01-30',
               '2024-01-31', '2024-02-01', '2024-02-02', '2024-02-03',
               '2024-02-04', '2024-02-05', '2024-02-06', '2024-02-07',
               '2024-02-08', '2024-02-09', '2024-02-10', '2024-02-11',
               '2024-02-12', '2024-02-13', '2024-02-14', '2024-02-15',
               '2024-02-16', '2024-02-17', '2024-02-18', '2024-02-19',
               '2024-02-20', '2024-02-21', '2024-02-22', '2024-02-23',
               '2024-02-24', '2024-02-25', '2024-02-26', '2024-02-27

In [140]:
mu = 18
sd = 4

s = pd.Series(
    np.random.normal(mu, sd, 90),
    index = date_idx
    )

s.head()

2024-01-03    24.672189
2024-01-04    20.949391
2024-01-05    17.193849
2024-01-06    17.396352
2024-01-07    21.664207
Freq: D, dtype: float64

In [141]:
s.describe()

count    90.000000
mean     18.200100
std       3.924567
min       7.520152
25%      15.820012
50%      18.291751
75%      21.394222
max      25.690387
dtype: float64

In [142]:
s.index

DatetimeIndex(['2024-01-03', '2024-01-04', '2024-01-05', '2024-01-06',
               '2024-01-07', '2024-01-08', '2024-01-09', '2024-01-10',
               '2024-01-11', '2024-01-12', '2024-01-13', '2024-01-14',
               '2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18',
               '2024-01-19', '2024-01-20', '2024-01-21', '2024-01-22',
               '2024-01-23', '2024-01-24', '2024-01-25', '2024-01-26',
               '2024-01-27', '2024-01-28', '2024-01-29', '2024-01-30',
               '2024-01-31', '2024-02-01', '2024-02-02', '2024-02-03',
               '2024-02-04', '2024-02-05', '2024-02-06', '2024-02-07',
               '2024-02-08', '2024-02-09', '2024-02-10', '2024-02-11',
               '2024-02-12', '2024-02-13', '2024-02-14', '2024-02-15',
               '2024-02-16', '2024-02-17', '2024-02-18', '2024-02-19',
               '2024-02-20', '2024-02-21', '2024-02-22', '2024-02-23',
               '2024-02-24', '2024-02-25', '2024-02-26', '2024-02-27',
      

In [143]:
s.dtype

dtype('float64')

In [144]:
np.random.seed(2024)

s[np.random.choice(date_idx, 10)] = np.nan
s

2024-01-03          NaN
2024-01-04    20.949391
2024-01-05    17.193849
2024-01-06    17.396352
2024-01-07    21.664207
                ...    
2024-03-28    20.141378
2024-03-29    22.328363
2024-03-30    18.370257
2024-03-31    15.458274
2024-04-01    20.367962
Freq: D, Length: 90, dtype: float64

In [145]:
np.random.seed(2024)

no_nan = s.dropna().index
no_nan_idx = np.random.choice(no_nan, 5)
print(no_nan_idx)

['2024-01-13T00:00:00.000000' '2024-01-04T00:00:00.000000'
 '2024-02-02T00:00:00.000000' '2024-02-12T00:00:00.000000'
 '2024-03-14T00:00:00.000000']


In [146]:
s[no_nan_idx] = s[no_nan_idx] * 3

In [147]:
s

2024-01-03          NaN
2024-01-04    62.848173
2024-01-05    17.193849
2024-01-06    17.396352
2024-01-07    21.664207
                ...    
2024-03-28    20.141378
2024-03-29    22.328363
2024-03-30    18.370257
2024-03-31    15.458274
2024-04-01    20.367962
Freq: D, Length: 90, dtype: float64

In [148]:
#2. 결측값 보정 방식 4가지 비교
na_dates = s.index[s.isnull()]
print("날짜 목록:\n", na_dates)

print("개수: ", len(na_dates))

날짜 목록:
 DatetimeIndex(['2024-01-03', '2024-01-11', '2024-01-30', '2024-02-08',
               '2024-02-13', '2024-02-27', '2024-03-03', '2024-03-05',
               '2024-03-08', '2024-03-17'],
              dtype='datetime64[us]', freq=None)
개수:  10


In [149]:
# 결측제거
no_nan = s.dropna()
no_nan

2024-01-04    62.848173
2024-01-05    17.193849
2024-01-06    17.396352
2024-01-07    21.664207
2024-01-08    22.641319
                ...    
2024-03-28    20.141378
2024-03-29    22.328363
2024-03-30    18.370257
2024-03-31    15.458274
2024-04-01    20.367962
Length: 80, dtype: float64

In [150]:
# 0으로 채움
zero_filled = s.fillna(0)
zero_filled

2024-01-03     0.000000
2024-01-04    62.848173
2024-01-05    17.193849
2024-01-06    17.396352
2024-01-07    21.664207
                ...    
2024-03-28    20.141378
2024-03-29    22.328363
2024-03-30    18.370257
2024-03-31    15.458274
2024-04-01    20.367962
Freq: D, Length: 90, dtype: float64

In [151]:
# 전체평균으로 채움
mean_filled = s.fillna(s.mean())
mean_filled

2024-01-03    20.788752
2024-01-04    62.848173
2024-01-05    17.193849
2024-01-06    17.396352
2024-01-07    21.664207
                ...    
2024-03-28    20.141378
2024-03-29    22.328363
2024-03-30    18.370257
2024-03-31    15.458274
2024-04-01    20.367962
Freq: D, Length: 90, dtype: float64

In [152]:
# 바로 앞 날짜값을 채움
prev_filled = s.ffill()
prev_filled

2024-01-03          NaN
2024-01-04    62.848173
2024-01-05    17.193849
2024-01-06    17.396352
2024-01-07    21.664207
                ...    
2024-03-28    20.141378
2024-03-29    22.328363
2024-03-30    18.370257
2024-03-31    15.458274
2024-04-01    20.367962
Freq: D, Length: 90, dtype: float64

In [153]:
df = pd.DataFrame(
    {
        "no_nan" : no_nan,
        "zero_filled" : zero_filled,
        "mean_filled" : mean_filled,
        "prev_filled" : prev_filled
    }
    )

summary = df.agg(["mean", "std"])
summary

,no_nan,zero_filled,mean_filled,prev_filled
mean,20.788752,18.478891,20.788752,20.855471
std,12.602628,13.569961,11.873525,12.961527


In [154]:
#가장 부적절한것은 0으로 채운것이다. 0으로 채웠기 때문에 outlier의 개수가 너무나도 많고, 평균값을 구할 때에도 0의 존재가 평균에 가장큰 영향을 준다.

In [155]:
# 3. 이상치 탐지와 보정

high = prev_filled.mean() + (2 * prev_filled.std())
low = prev_filled.mean() - (2 * prev_filled.std())

print("상한: ",  high)
print("하한: ",  low)

상한:  46.77852502823547
하한:  -5.067583619771419


In [156]:
print(prev_filled[(prev_filled < low) | (prev_filled > high)])

print(no_nan_idx)

#날짜가 하나 더있다. 2024-02-13 과 2024-02-12가 값이 같은걸보니 ffill()로 채워진 수치로보임.

2024-01-04    62.848173
2024-01-13    66.642633
2024-02-02    62.270870
2024-02-12    65.627725
2024-02-13    65.627725
2024-03-14    77.071161
dtype: float64
['2024-01-13T00:00:00.000000' '2024-01-04T00:00:00.000000'
 '2024-02-02T00:00:00.000000' '2024-02-12T00:00:00.000000'
 '2024-03-14T00:00:00.000000']


In [157]:
clean = pd.Series(np.where(prev_filled > high, high,
                           np.where(prev_filled < low, low, prev_filled)),
                           index = prev_filled.index
        )
clean

2024-01-03          NaN
2024-01-04    46.778525
2024-01-05    17.193849
2024-01-06    17.396352
2024-01-07    21.664207
                ...    
2024-03-28    20.141378
2024-03-29    22.328363
2024-03-30    18.370257
2024-03-31    15.458274
2024-04-01    20.367962
Freq: D, Length: 90, dtype: float64

In [158]:
prev_filled.describe()

count    89.000000
mean     20.855471
std      12.961527
min       7.520152
25%      15.458274
50%      17.497437
75%      21.235104
max      77.071161
dtype: float64

In [159]:
clean.describe()

count    89.000000
mean     19.513705
std       8.196107
min       7.520152
25%      15.458274
50%      17.497437
75%      21.235104
max      46.778525
dtype: float64

In [160]:
# mean, std, max가 하락했다. 하한보다 수치가 적은 수치가 없고, 상한보다 수치가 높은 수치가 바뀌었기 때문에 mean, std, max가 내려갔다.

In [161]:
# 4. 월별 집계와 이동평균

print(clean.groupby(clean.index.month).agg(["count", "mean", "min", "max"]))

   count       mean        min        max
1     28  19.433225   7.520152  46.778525
2     29  20.924164  12.277801  46.778525
3     31  18.239378   8.452445  46.778525
4      1  20.367962  20.367962  20.367962


In [162]:
clean.rolling(7).mean()

#7개의 행이 없기 때문에 nan으로 나온다.

2024-01-03          NaN
2024-01-04          NaN
2024-01-05          NaN
2024-01-06          NaN
2024-01-07          NaN
                ...    
2024-03-28    18.864034
2024-03-29    18.949665
2024-03-30    18.719565
2024-03-31    18.812076
2024-04-01    19.008099
Freq: D, Length: 90, dtype: float64

In [163]:
print(clean.sort_values(ascending=False).head(5))

2024-01-04    46.778525
2024-01-13    46.778525
2024-02-13    46.778525
2024-02-12    46.778525
2024-02-02    46.778525
dtype: float64


In [164]:
print(len(clean[clean > clean.mean()]))

33


# 2. 300건 규모의 온라인 쇼핑몰 주문 데이터를 생성해 할인·배송비 정책을 반영한 정산표 만들기

In [165]:
np.random.seed(7)
order_no = 300

ranks = ["일반", "실버", "골드", "VIP"]
category = ["식품", "의류", "가전", "도서", "뷰티"]
price = [5000, 12000, 25000, 48000, 99000]

df = pd.DataFrame(
    {
        "주문번호" : ["ORD" + str(i).zfill(4) for i in range(1, order_no + 1)],
        "주문일" : [np.datetime64("2024-01-01") + pd.DateOffset(i) for i in np.random.randint(0, 180, order_no)],
        "회원등급" : np.random.choice(ranks, order_no, p=[0.5, 0.25, 0.15, 0.1]),
        "카테고리" : np.random.choice(category, order_no),
        "수량" : np.random.randint(1, 6, order_no),
        "단가" : np.random.choice(price, order_no),
        "쿠폰사용" : np.random.choice([True, False], order_no, p=[0.3, 0.7])
    }
)

In [166]:
df.head()

,주문번호,주문일,회원등급,카테고리,수량,단가,쿠폰사용
0,ORD0001,2024-06-24,일반,도서,4,12000,True
1,ORD0002,2024-01-26,일반,식품,3,5000,False
2,ORD0003,2024-03-08,일반,식품,1,12000,True
3,ORD0004,2024-05-31,실버,의류,1,12000,True
4,ORD0005,2024-04-13,일반,가전,4,12000,False


In [167]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   주문번호    300 non-null    str          
 1   주문일     300 non-null    datetime64[s]
 2   회원등급    300 non-null    str          
 3   카테고리    300 non-null    str          
 4   수량      300 non-null    int32        
 5   단가      300 non-null    int64        
 6   쿠폰사용    300 non-null    bool         
dtypes: bool(1), datetime64[s](1), int32(1), int64(1), str(3)
memory usage: 13.3 KB


In [168]:
df.describe()

,주문일,수량,단가
count,300,300.000000,300.000000
mean,2024-03-31 09:45:36,2.813333,34856.666667
min,2024-01-01 00:00:00,1.000000,5000.000000
25%,2024-02-15 00:00:00,1.000000,12000.000000
50%,2024-04-01 12:00:00,3.000000,25000.000000
75%,2024-05-16 00:00:00,4.000000,48000.000000
max,2024-06-28 00:00:00,5.000000,99000.000000
std,NaN,1.457935,31671.645035


In [169]:
df["회원등급"].value_counts()

# 50% = 150, 25% = 75, 15% = 45, 10% = 30
# 비슷하다.

회원등급
일반     154
실버      63
골드      47
VIP     36
Name: count, dtype: int64

In [170]:
np.random.seed(7)

rand_nan = np.random.choice(df.index, 20, replace = False)
df.loc[rand_nan, "수량"] = np.nan

df

,주문번호,주문일,회원등급,카테고리,수량,단가,쿠폰사용
0,ORD0001,2024-06-24,일반,도서,4.0,12000,True
1,ORD0002,2024-01-26,일반,식품,3.0,5000,False
2,ORD0003,2024-03-08,일반,식품,1.0,12000,True
3,ORD0004,2024-05-31,실버,의류,NaN,12000,True
4,ORD0005,2024-04-13,일반,가전,4.0,12000,False
...,...,...,...,...,...,...,...
295,ORD0296,2024-05-10,VIP,식품,4.0,48000,False
296,ORD0297,2024-02-07,실버,의류,5.0,99000,True
297,ORD0298,2024-03-27,VIP,식품,NaN,48000,True
298,ORD0299,2024-02-16,골드,뷰티,1.0,48000,False


In [171]:
df["수량"].isnull().sum()

np.int64(20)

In [172]:
df["수량"] = df["수량"].fillna(df["수량"].median())

In [173]:
# 1. 정산 로직을 파생 열로 구현

rate = {"일반" : 0,
        "실버" : 3,
        "골드" : 5,
        "VIP" : 10}

def func(x):
    if x >= 200000:
        return "대형"
    elif x >= 50000:
        return "중형"
    return "소형"

df["주문금액"] = df["수량"] * df["단가"]
df["등급활인율"] = df["회원등급"].map(rate)
df["쿠폰활인율"] = np.where(df["쿠폰사용"], 5, 0)
df["총활인율"] = df["등급활인율"] + df["쿠폰활인율"]
df["활인금액"] = round(df["주문금액"] * df["총활인율"] / 100)
df["배송비"] = np.where(((df["주문금액"] - df["활인금액"]) >= 30000), 0, 3000)
df["최종결제금액"] = df["주문금액"] - df["활인금액"] + df["배송비"]
df["주문규모"] = df["최종결제금액"].apply(func)

In [174]:
df

,주문번호,주문일,회원등급,카테고리,수량,단가,쿠폰사용,주문금액,등급활인율,쿠폰활인율,총활인율,활인금액,배송비,최종결제금액,주문규모
0,ORD0001,2024-06-24,일반,도서,4.0,12000,True,48000.0,0,5,5,2400.0,0,45600.0,소형
1,ORD0002,2024-01-26,일반,식품,3.0,5000,False,15000.0,0,0,0,0.0,3000,18000.0,소형
2,ORD0003,2024-03-08,일반,식품,1.0,12000,True,12000.0,0,5,5,600.0,3000,14400.0,소형
3,ORD0004,2024-05-31,실버,의류,3.0,12000,True,36000.0,3,5,8,2880.0,0,33120.0,소형
4,ORD0005,2024-04-13,일반,가전,4.0,12000,False,48000.0,0,0,0,0.0,0,48000.0,소형
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,ORD0296,2024-05-10,VIP,식품,4.0,48000,False,192000.0,10,0,10,19200.0,0,172800.0,중형
296,ORD0297,2024-02-07,실버,의류,5.0,99000,True,495000.0,3,5,8,39600.0,0,455400.0,대형
297,ORD0298,2024-03-27,VIP,식품,3.0,48000,True,144000.0,10,5,15,21600.0,0,122400.0,중형
298,ORD0299,2024-02-16,골드,뷰티,1.0,48000,False,48000.0,5,0,5,2400.0,0,45600.0,소형


In [175]:
# 2. 집계와 조건 분석

df["월"] = df["주문일"].dt.month
df

,주문번호,주문일,회원등급,카테고리,수량,단가,쿠폰사용,주문금액,등급활인율,쿠폰활인율,총활인율,활인금액,배송비,최종결제금액,주문규모,월
0,ORD0001,2024-06-24,일반,도서,4.0,12000,True,48000.0,0,5,5,2400.0,0,45600.0,소형,6
1,ORD0002,2024-01-26,일반,식품,3.0,5000,False,15000.0,0,0,0,0.0,3000,18000.0,소형,1
2,ORD0003,2024-03-08,일반,식품,1.0,12000,True,12000.0,0,5,5,600.0,3000,14400.0,소형,3
3,ORD0004,2024-05-31,실버,의류,3.0,12000,True,36000.0,3,5,8,2880.0,0,33120.0,소형,5
4,ORD0005,2024-04-13,일반,가전,4.0,12000,False,48000.0,0,0,0,0.0,0,48000.0,소형,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,ORD0296,2024-05-10,VIP,식품,4.0,48000,False,192000.0,10,0,10,19200.0,0,172800.0,중형,5
296,ORD0297,2024-02-07,실버,의류,5.0,99000,True,495000.0,3,5,8,39600.0,0,455400.0,대형,2
297,ORD0298,2024-03-27,VIP,식품,3.0,48000,True,144000.0,10,5,15,21600.0,0,122400.0,중형,3
298,ORD0299,2024-02-16,골드,뷰티,1.0,48000,False,48000.0,5,0,5,2400.0,0,45600.0,소형,2


In [176]:
df.groupby("월")["최종결제금액"].agg(["count", "sum", "mean"])

,count,sum,mean
월,,,
1,46,4668650.0,101492.391304
2,54,4861220.0,90022.592593
3,48,5188410.0,108091.875000
4,50,5536030.0,110720.600000
5,56,4069880.0,72676.428571
6,46,4597590.0,99947.608696


In [177]:
df.groupby(["카테고리", "회원등급"])["최종결제금액"].sum()

카테고리  회원등급
가전    VIP      530350.0
      골드       626500.0
      실버      1581650.0
      일반      3126600.0
도서    VIP      601200.0
      골드       622750.0
      실버      1315360.0
      일반      3196900.0
뷰티    VIP      464250.0
      골드       425550.0
      실버       653610.0
      일반      2603750.0
식품    VIP      609600.0
      골드      1261050.0
      실버      2161110.0
      일반      3893050.0
의류    VIP      928600.0
      골드       660700.0
      실버      1438850.0
      일반      2220350.0
Name: 최종결제금액, dtype: float64

In [178]:
df[(df["회원등급"] == "VIP") & (df["최종결제금액"] >= 100000)]

,주문번호,주문일,회원등급,카테고리,수량,단가,쿠폰사용,주문금액,등급활인율,쿠폰활인율,총활인율,활인금액,배송비,최종결제금액,주문규모,월
41,ORD0042,2024-06-08,VIP,의류,2.0,99000,False,198000.0,10,0,10,19800.0,0,178200.0,중형,6
54,ORD0055,2024-04-10,VIP,가전,4.0,48000,True,192000.0,10,5,15,28800.0,0,163200.0,중형,4
81,ORD0082,2024-03-17,VIP,의류,3.0,99000,False,297000.0,10,0,10,29700.0,0,267300.0,대형,3
121,ORD0122,2024-04-03,VIP,의류,4.0,48000,True,192000.0,10,5,15,28800.0,0,163200.0,중형,4
176,ORD0177,2024-03-26,VIP,뷰티,5.0,25000,False,125000.0,10,0,10,12500.0,0,112500.0,중형,3
193,ORD0194,2024-06-28,VIP,가전,3.0,48000,False,144000.0,10,0,10,14400.0,0,129600.0,중형,6
194,ORD0195,2024-02-24,VIP,의류,5.0,25000,False,125000.0,10,0,10,12500.0,0,112500.0,중형,2
206,ORD0207,2024-06-20,VIP,식품,2.0,99000,False,198000.0,10,0,10,19800.0,0,178200.0,중형,6
265,ORD0266,2024-02-13,VIP,뷰티,3.0,48000,False,144000.0,10,0,10,14400.0,0,129600.0,중형,2
284,ORD0285,2024-02-24,VIP,도서,4.0,48000,False,192000.0,10,0,10,19200.0,0,172800.0,중형,2


In [179]:
df[(df["쿠폰사용"]) & (df["카테고리"] == "가전")]

,주문번호,주문일,회원등급,카테고리,수량,단가,쿠폰사용,주문금액,등급활인율,쿠폰활인율,총활인율,활인금액,배송비,최종결제금액,주문규모,월
12,ORD0013,2024-05-16,일반,가전,5.0,48000,True,240000.0,0,5,5,12000.0,0,228000.0,대형,5
14,ORD0015,2024-03-09,일반,가전,4.0,48000,True,192000.0,0,5,5,9600.0,0,182400.0,중형,3
40,ORD0041,2024-04-28,일반,가전,4.0,48000,True,192000.0,0,5,5,9600.0,0,182400.0,중형,4
54,ORD0055,2024-04-10,VIP,가전,4.0,48000,True,192000.0,10,5,15,28800.0,0,163200.0,중형,4
72,ORD0073,2024-01-22,일반,가전,4.0,12000,True,48000.0,0,5,5,2400.0,0,45600.0,소형,1
85,ORD0086,2024-05-04,일반,가전,1.0,5000,True,5000.0,0,5,5,250.0,3000,7750.0,소형,5
100,ORD0101,2024-05-20,골드,가전,1.0,25000,True,25000.0,5,5,10,2500.0,3000,25500.0,소형,5
128,ORD0129,2024-06-24,일반,가전,1.0,12000,True,12000.0,0,5,5,600.0,3000,14400.0,소형,6
130,ORD0131,2024-03-28,일반,가전,1.0,99000,True,99000.0,0,5,5,4950.0,0,94050.0,중형,3
145,ORD0146,2024-04-19,VIP,가전,3.0,25000,True,75000.0,10,5,15,11250.0,0,63750.0,중형,4


In [180]:
df.sort_values(["카테고리", "최종결제금액"], ascending=False).groupby("카테고리").head(2)

,주문번호,주문일,회원등급,카테고리,수량,단가,쿠폰사용,주문금액,등급활인율,쿠폰활인율,총활인율,활인금액,배송비,최종결제금액,주문규모,월
296,ORD0297,2024-02-07,실버,의류,5.0,99000,True,495000.0,3,5,8,39600.0,0,455400.0,대형,2
213,ORD0214,2024-02-10,실버,의류,4.0,99000,False,396000.0,3,0,3,11880.0,0,384120.0,대형,2
7,ORD0008,2024-01-24,일반,식품,5.0,99000,False,495000.0,0,0,0,0.0,0,495000.0,대형,1
170,ORD0171,2024-02-04,실버,식품,5.0,99000,False,495000.0,3,0,3,14850.0,0,480150.0,대형,2
70,ORD0071,2024-06-27,일반,뷰티,5.0,99000,False,495000.0,0,0,0,0.0,0,495000.0,대형,6
267,ORD0268,2024-01-19,일반,뷰티,4.0,99000,True,396000.0,0,5,5,19800.0,0,376200.0,대형,1
163,ORD0164,2024-06-17,일반,도서,4.0,99000,False,396000.0,0,0,0,0.0,0,396000.0,대형,6
195,ORD0196,2024-02-12,일반,도서,4.0,99000,True,396000.0,0,5,5,19800.0,0,376200.0,대형,2
56,ORD0057,2024-03-08,일반,가전,5.0,99000,False,495000.0,0,0,0,0.0,0,495000.0,대형,3
215,ORD0216,2024-03-10,일반,가전,5.0,99000,False,495000.0,0,0,0,0.0,0,495000.0,대형,3


In [181]:
# 4. 정리와 저장

df.drop(["등급활인율", "쿠폰활인율"], axis = 1, inplace = True)

In [182]:
df.to_csv("orders.csv", encoding = "utf-8-sig")

In [183]:
df2 = pd.read_csv("orders.csv", index_col = "주문번호" , parse_dates=["주문일"])

In [184]:
df2.loc['ORD0010']

Unnamed: 0                      9
주문일           2024-03-30 00:00:00
회원등급                           실버
카테고리                           도서
수량                            4.0
단가                          48000
쿠폰사용                        False
주문금액                     192000.0
총활인율                            3
활인금액                       5760.0
배송비                             0
최종결제금액                   186240.0
주문규모                           중형
월                               3
Name: ORD0010, dtype: object

In [185]:
df2.iloc[9]

Unnamed: 0                      9
주문일           2024-03-30 00:00:00
회원등급                           실버
카테고리                           도서
수량                            4.0
단가                          48000
쿠폰사용                        False
주문금액                     192000.0
총활인율                            3
활인금액                       5760.0
배송비                             0
최종결제금액                   186240.0
주문규모                           중형
월                               3
Name: ORD0010, dtype: object

In [186]:
# df2를 읽을 때 주문번호를 index로 했기때문에 loc과 주문번호를 사용해 찾을 수 있고, iloc으로 9번째 행을 가져왔기 때문에 같은 값이 나온다.

# 3. 4개 분기 지점별 실적 데이터를 생성·연결해 MultiIndex 집계와 성장률 리포트 만들기

In [187]:
# 1. 분기 데이터 생성 함수 만들고 연결하기
np.random.seed(99)

fp = pd.MultiIndex.from_product(
    [["서울", "부산", "대구", "광주", "대전"], ["A", "B", "C"]],
    names=["지점", "상품"]
)

fp

MultiIndex([('서울', 'A'),
            ('서울', 'B'),
            ('서울', 'C'),
            ('부산', 'A'),
            ('부산', 'B'),
            ('부산', 'C'),
            ('대구', 'A'),
            ('대구', 'B'),
            ('대구', 'C'),
            ('광주', 'A'),
            ('광주', 'B'),
            ('광주', 'C'),
            ('대전', 'A'),
            ('대전', 'B'),
            ('대전', 'C')],
           names=['지점', '상품'])

In [188]:
def make_quarter(q):
    idx = pd.MultiIndex.from_product(
    [["서울", "부산", "대구", "광주", "대전"], ["A", "B", "C"]],
    names=["지점", "상품"]
    )
    return pd.DataFrame(
        {
            "판매량" : np.random.randint(50, 300, len(idx)),
            "단가" : np.random.choice([10000, 15000, 20000], len(idx)),
            "반품" : np.random.randint(0, 20, len(idx)),
            "분기" : q
        },
        index = idx
    )

In [204]:
np.random.seed(99)

qs = ["1Q", "2Q", "3Q", "4Q"]

sales = pd.concat([make_quarter(q) for q in qs], axis = 0)
sales

판매량     단가  반품  분기
지점 상품                    
서울 A   179  15000  14  1Q
   B    85  10000   4  1Q
   C   235  15000  12  1Q
부산 A   218  20000  17  1Q
   B   251  10000   9  1Q
   C   282  20000   0  1Q
대구 A   260  10000  15  1Q
   B   118  10000   9  1Q
   C   247  10000   3  1Q
광주 A   230  10000  19  1Q
   B   179  10000  18  1Q
   C   201  15000   5  1Q
대전 A    85  20000  12  1Q
   B   105  20000   0  1Q
   C   291  15000   1  1Q
서울 A   222  10000   8  2Q
   B   125  15000   3  2Q
   C   164  20000  13  2Q
부산 A    61  15000   9  2Q
   B   271  10000  12  2Q
   C   145  10000   4  2Q
대구 A   163  15000   1  2Q
   B    91  20000  13  2Q
   C   173  10000  11  2Q
광주 A   211  20000  11  2Q
   B    62  15000  12  2Q
   C   127  10000   4  2Q
대전 A   254  20000   9  2Q
   B   277  20000   5  2Q
   C   138  20000  10  2Q
서울 A   141  10000  16  3Q
   B   215  20000  18  3Q
   C   210  20000  10  3Q
부산 A   181  10000  10  3Q
   B   185  10000  12  3Q
   C    78  10000   1  3Q
대구 A   223  10000  18  3Q
   B   237  20000   6  3Q
   C   283  15000  18  3Q
광주 A   119  20000  13  3Q
   B   223  10000   2  3Q
   C   217  10000   7  3Q
대전 A   105  10000   2  3Q
   B   234  15000   3  3Q
   C   170  15000  15  3Q
서울 A   144  20000   2  4Q
   B   207  10000   7  4Q
   C   124  20000   0  4Q
부산 A    84  10000   6  4Q
   B   268  10000  18  4Q
   C   248  15000   8  4Q
대구 A   211  15000   4  4Q
   B   189  20000  18  4Q
   C   144  15000  13  4Q
광주 A   240  15000  15  4Q
   B   120  15000   0  4Q
   C   193  20000   1  4Q
대전 A   177  15000  16  4Q
   B   273  15000  11  4Q
   C   117  10000   3  4Q

In [191]:
print("number of rows: ", len(sales))

number of rows:  60


In [192]:
sales.shape

(60, 4)

In [193]:
sales.index.names

FrozenList(['지점', '상품'])

In [194]:
sales.head()

판매량     단가  반품  분기
지점 상품                    
서울 A   179  15000  14  1Q
   B    85  10000   4  1Q
   C   235  15000  12  1Q
부산 A   218  20000  17  1Q
   B   251  10000   9  1Q

In [209]:
np.random.seed(99)

sales['판매량'] = sales['판매량'].astype(float)
nan_idx = np.random.choice(len(sales), 8, replace = False)
sales.iloc[nan_idx, sales.columns.get_loc("판매량")] = np.nan
sales['판매량'] = sales['판매량'].fillna(sales.groupby("지점")['판매량'].transform("median"))
sales

판매량     단가  반품  분기
지점 상품                      
서울 A   179.0  15000  14  1Q
   B    85.0  10000   4  1Q
   C   235.0  15000  12  1Q
부산 A   218.0  20000  17  1Q
   B   251.0  10000   9  1Q
   C   282.0  20000   0  1Q
대구 A   192.0  10000  15  1Q
   B   192.0  10000   9  1Q
   C   247.0  10000   3  1Q
광주 A   230.0  10000  19  1Q
   B   179.0  10000  18  1Q
   C   201.0  15000   5  1Q
대전 A    85.0  20000  12  1Q
   B   105.0  20000   0  1Q
   C   170.0  15000   1  1Q
서울 A   222.0  10000   8  2Q
   B   171.5  15000   3  2Q
   C   164.0  20000  13  2Q
부산 A    61.0  15000   9  2Q
   B   271.0  10000  12  2Q
   C   145.0  10000   4  2Q
대구 A   163.0  15000   1  2Q
   B    91.0  20000  13  2Q
   C   173.0  10000  11  2Q
광주 A   193.0  20000  11  2Q
   B    62.0  15000  12  2Q
   C   127.0  10000   4  2Q
대전 A   254.0  20000   9  2Q
   B   277.0  20000   5  2Q
   C   138.0  20000  10  2Q
서울 A   141.0  10000  16  3Q
   B   171.5  20000  18  3Q
   C   210.0  20000  10  3Q
부산 A   181.0  10000  10  3Q
   B   185.0  10000  12  3Q
   C    78.0  10000   1  3Q
대구 A   192.0  10000  18  3Q
   B   237.0  20000   6  3Q
   C   283.0  15000  18  3Q
광주 A   119.0  20000  13  3Q
   B   223.0  10000   2  3Q
   C   217.0  10000   7  3Q
대전 A   105.0  10000   2  3Q
   B   234.0  15000   3  3Q
   C   170.0  15000  15  3Q
서울 A   144.0  20000   2  4Q
   B   207.0  10000   7  4Q
   C   124.0  20000   0  4Q
부산 A    84.0  10000   6  4Q
   B   268.0  10000  18  4Q
   C   248.0  15000   8  4Q
대구 A   211.0  15000   4  4Q
   B   192.0  20000  18  4Q
   C   144.0  15000  13  4Q
광주 A   240.0  15000  15  4Q
   B   120.0  15000   0  4Q
   C   193.0  20000   1  4Q
대전 A   177.0  15000  16  4Q
   B   273.0  15000  11  4Q
   C   117.0  10000   3  4Q

In [220]:
# 2. 파생 열과 MultiIndex 집계

sales["매출"] = (sales["판매량"] * sales["단가"]).round(2)
sales["반품율"] = ((sales["반품"] / sales["판매량"]) * 100).round(2)
sales["등급"] = np.where(sales["매출"] >= 4000000, "A",
                        np.where(sales["매출"] >= 2000000, "B", "C"))
sales

판매량     단가  반품  분기         매출    반품율 등급
지점 상품                                           
서울 A   179.0  15000  14  1Q  2685000.0   7.82  B
   B    85.0  10000   4  1Q   850000.0   4.71  C
   C   235.0  15000  12  1Q  3525000.0   5.11  B
부산 A   218.0  20000  17  1Q  4360000.0   7.80  A
   B   251.0  10000   9  1Q  2510000.0   3.59  B
   C   282.0  20000   0  1Q  5640000.0   0.00  A
대구 A   192.0  10000  15  1Q  1920000.0   7.81  C
   B   192.0  10000   9  1Q  1920000.0   4.69  C
   C   247.0  10000   3  1Q  2470000.0   1.21  B
광주 A   230.0  10000  19  1Q  2300000.0   8.26  B
   B   179.0  10000  18  1Q  1790000.0  10.06  C
   C   201.0  15000   5  1Q  3015000.0   2.49  B
대전 A    85.0  20000  12  1Q  1700000.0  14.12  C
   B   105.0  20000   0  1Q  2100000.0   0.00  B
   C   170.0  15000   1  1Q  2550000.0   0.59  B
서울 A   222.0  10000   8  2Q  2220000.0   3.60  B
   B   171.5  15000   3  2Q  2572500.0   1.75  B
   C   164.0  20000  13  2Q  3280000.0   7.93  B
부산 A    61.0  15000   9  2Q   915000.0  14.75  C
   B   271.0  10000  12  2Q  2710000.0   4.43  B
   C   145.0  10000   4  2Q  1450000.0   2.76  C
대구 A   163.0  15000   1  2Q  2445000.0   0.61  B
   B    91.0  20000  13  2Q  1820000.0  14.29  C
   C   173.0  10000  11  2Q  1730000.0   6.36  C
광주 A   193.0  20000  11  2Q  3860000.0   5.70  B
   B    62.0  15000  12  2Q   930000.0  19.35  C
   C   127.0  10000   4  2Q  1270000.0   3.15  C
대전 A   254.0  20000   9  2Q  5080000.0   3.54  A
   B   277.0  20000   5  2Q  5540000.0   1.81  A
   C   138.0  20000  10  2Q  2760000.0   7.25  B
서울 A   141.0  10000  16  3Q  1410000.0  11.35  C
   B   171.5  20000  18  3Q  3430000.0  10.50  B
   C   210.0  20000  10  3Q  4200000.0   4.76  A
부산 A   181.0  10000  10  3Q  1810000.0   5.52  C
   B   185.0  10000  12  3Q  1850000.0   6.49  C
   C    78.0  10000   1  3Q   780000.0   1.28  C
대구 A   192.0  10000  18  3Q  1920000.0   9.38  C
   B   237.0  20000   6  3Q  4740000.0   2.53  A
   C   283.0  15000  18  3Q  4245000.0   6.36  A
광주 A   119.0  20000  13  3Q  2380000.0  10.92  B
   B   223.0  10000   2  3Q  2230000.0   0.90  B
   C   217.0  10000   7  3Q  2170000.0   3.23  B
대전 A   105.0  10000   2  3Q  1050000.0   1.90  C
   B   234.0  15000   3  3Q  3510000.0   1.28  B
   C   170.0  15000  15  3Q  2550000.0   8.82  B
서울 A   144.0  20000   2  4Q  2880000.0   1.39  B
   B   207.0  10000   7  4Q  2070000.0   3.38  B
   C   124.0  20000   0  4Q  2480000.0   0.00  B
부산 A    84.0  10000   6  4Q   840000.0   7.14  C
   B   268.0  10000  18  4Q  2680000.0   6.72  B
   C   248.0  15000   8  4Q  3720000.0   3.23  B
대구 A   211.0  15000   4  4Q  3165000.0   1.90  B
   B   192.0  20000  18  4Q  3840000.0   9.38  B
   C   144.0  15000  13  4Q  2160000.0   9.03  B
광주 A   240.0  15000  15  4Q  3600000.0   6.25  B
   B   120.0  15000   0  4Q  1800000.0   0.00  C
   C   193.0  20000   1  4Q  3860000.0   0.52  B
대전 A   177.0  15000  16  4Q  2655000.0   9.04  B
   B   273.0  15000  11  4Q  4095000.0   4.03  A
   C   117.0  10000   3  4Q  1170000.0   2.56  C

In [226]:
sales.groupby("지점")["매출"].agg(["count", "sum", "mean", "max"])

,count,sum,mean,max
지점,,,,
광주,12,29205000.0,2.433750e+06,3860000.0
대구,12,32375000.0,2.697917e+06,4740000.0
대전,12,34760000.0,2.896667e+06,5540000.0
부산,12,29265000.0,2.438750e+06,5640000.0
서울,12,31602500.0,2.633542e+06,4200000.0


In [227]:
sales.groupby(["지점", "상품"])["매출"].sum()

지점  상품
광주  A     12140000.0
    B      6750000.0
    C     10315000.0
대구  A      9450000.0
    B     12320000.0
    C     10605000.0
대전  A     10485000.0
    B     15245000.0
    C      9030000.0
부산  A      7925000.0
    B      9750000.0
    C     11590000.0
서울  A      9195000.0
    B      8922500.0
    C     13485000.0
Name: 매출, dtype: float64

In [228]:
sales.groupby(["분기", "지점"])["매출"].sum()

분기  지점
1Q  광주     7105000.0
    대구     6310000.0
    대전     6350000.0
    부산    12510000.0
    서울     7060000.0
2Q  광주     6060000.0
    대구     5995000.0
    대전    13380000.0
    부산     5075000.0
    서울     8072500.0
3Q  광주     6780000.0
    대구    10905000.0
    대전     7110000.0
    부산     4440000.0
    서울     9040000.0
4Q  광주     9260000.0
    대구     9165000.0
    대전     7920000.0
    부산     7240000.0
    서울     7430000.0
Name: 매출, dtype: float64

In [230]:
sales.groupby("등급").size()

등급
A     8
B    32
C    20
dtype: int64

In [236]:
# 3. 순위와 교차표(피벗)

annual_sales = sales.groupby("지점")["매출"].sum().sort_values(ascending=False)
annual_sales

지점
대전    34760000.0
대구    32375000.0
서울    31602500.0
부산    29265000.0
광주    29205000.0
Name: 매출, dtype: float64

In [238]:
annual_sales.rank()

지점
대전    5.0
대구    4.0
서울    3.0
부산    2.0
광주    1.0
Name: 매출, dtype: float64

In [246]:
pv = pd.pivot_table(sales, values = "매출", index = "지점", columns = "분기", aggfunc = "sum")
pv["연간합계"] = pv.sum(axis=1)
pv["성장률"] = ((pv["4Q"] - pv["1Q"]) / pv["1Q"] * 100).round(1)
pv = pv.sort_values(by="성장률", ascending=False)
pv

분기,1Q,2Q,3Q,4Q,연간합계,성장률
지점,,,,,,
대구,6310000.0,5995000.0,10905000.0,9165000.0,32375000.0,45.2
광주,7105000.0,6060000.0,6780000.0,9260000.0,29205000.0,30.3
대전,6350000.0,13380000.0,7110000.0,7920000.0,34760000.0,24.7
서울,7060000.0,8072500.0,9040000.0,7430000.0,31602500.0,5.2
부산,12510000.0,5075000.0,4440000.0,7240000.0,29265000.0,-42.1


In [250]:
# 4. 가로 방향 연결과 저장

q1 = sales[sales["분기"] == "1Q"].groupby("지점")["매출"].sum()
q4 = sales[sales["분기"] == "4Q"].groupby("지점")["매출"].sum()
conc = pd.concat([q1, q4], axis = 1, keys = ["1Q", "4Q"])
conc

,1Q,4Q
지점,,
광주,7105000.0,9260000.0
대구,6310000.0,9165000.0
대전,6350000.0,7920000.0
부산,12510000.0,7240000.0
서울,7060000.0,7430000.0


In [255]:
no_one = sales.sort_values(by="매출", ascending=False).groupby("지점").head(1)
no_one

,,판매량,단가,반품,분기,매출,반품율,등급
지점,상품,,,,,,,
부산,C,282.0,20000,0,1Q,5640000.0,0.00,A
대전,B,277.0,20000,5,2Q,5540000.0,1.81,A
대구,B,237.0,20000,6,3Q,4740000.0,2.53,A
서울,C,210.0,20000,10,3Q,4200000.0,4.76,A
광주,C,193.0,20000,1,4Q,3860000.0,0.52,B


In [259]:
sales.to_csv("sales_report.csv", encoding = "utf-8-sig")

In [260]:
df2 = pd.read_csv("sales_report.csv", index_col = ["지점", "상품"])
df2

판매량     단가  반품  분기         매출    반품율 등급
지점 상품                                           
서울 A   179.0  15000  14  1Q  2685000.0   7.82  B
   B    85.0  10000   4  1Q   850000.0   4.71  C
   C   235.0  15000  12  1Q  3525000.0   5.11  B
부산 A   218.0  20000  17  1Q  4360000.0   7.80  A
   B   251.0  10000   9  1Q  2510000.0   3.59  B
   C   282.0  20000   0  1Q  5640000.0   0.00  A
대구 A   192.0  10000  15  1Q  1920000.0   7.81  C
   B   192.0  10000   9  1Q  1920000.0   4.69  C
   C   247.0  10000   3  1Q  2470000.0   1.21  B
광주 A   230.0  10000  19  1Q  2300000.0   8.26  B
   B   179.0  10000  18  1Q  1790000.0  10.06  C
   C   201.0  15000   5  1Q  3015000.0   2.49  B
대전 A    85.0  20000  12  1Q  1700000.0  14.12  C
   B   105.0  20000   0  1Q  2100000.0   0.00  B
   C   170.0  15000   1  1Q  2550000.0   0.59  B
서울 A   222.0  10000   8  2Q  2220000.0   3.60  B
   B   171.5  15000   3  2Q  2572500.0   1.75  B
   C   164.0  20000  13  2Q  3280000.0   7.93  B
부산 A    61.0  15000   9  2Q   915000.0  14.75  C
   B   271.0  10000  12  2Q  2710000.0   4.43  B
   C   145.0  10000   4  2Q  1450000.0   2.76  C
대구 A   163.0  15000   1  2Q  2445000.0   0.61  B
   B    91.0  20000  13  2Q  1820000.0  14.29  C
   C   173.0  10000  11  2Q  1730000.0   6.36  C
광주 A   193.0  20000  11  2Q  3860000.0   5.70  B
   B    62.0  15000  12  2Q   930000.0  19.35  C
   C   127.0  10000   4  2Q  1270000.0   3.15  C
대전 A   254.0  20000   9  2Q  5080000.0   3.54  A
   B   277.0  20000   5  2Q  5540000.0   1.81  A
   C   138.0  20000  10  2Q  2760000.0   7.25  B
서울 A   141.0  10000  16  3Q  1410000.0  11.35  C
   B   171.5  20000  18  3Q  3430000.0  10.50  B
   C   210.0  20000  10  3Q  4200000.0   4.76  A
부산 A   181.0  10000  10  3Q  1810000.0   5.52  C
   B   185.0  10000  12  3Q  1850000.0   6.49  C
   C    78.0  10000   1  3Q   780000.0   1.28  C
대구 A   192.0  10000  18  3Q  1920000.0   9.38  C
   B   237.0  20000   6  3Q  4740000.0   2.53  A
   C   283.0  15000  18  3Q  4245000.0   6.36  A
광주 A   119.0  20000  13  3Q  2380000.0  10.92  B
   B   223.0  10000   2  3Q  2230000.0   0.90  B
   C   217.0  10000   7  3Q  2170000.0   3.23  B
대전 A   105.0  10000   2  3Q  1050000.0   1.90  C
   B   234.0  15000   3  3Q  3510000.0   1.28  B
   C   170.0  15000  15  3Q  2550000.0   8.82  B
서울 A   144.0  20000   2  4Q  2880000.0   1.39  B
   B   207.0  10000   7  4Q  2070000.0   3.38  B
   C   124.0  20000   0  4Q  2480000.0   0.00  B
부산 A    84.0  10000   6  4Q   840000.0   7.14  C
   B   268.0  10000  18  4Q  2680000.0   6.72  B
   C   248.0  15000   8  4Q  3720000.0   3.23  B
대구 A   211.0  15000   4  4Q  3165000.0   1.90  B
   B   192.0  20000  18  4Q  3840000.0   9.38  B
   C   144.0  15000  13  4Q  2160000.0   9.03  B
광주 A   240.0  15000  15  4Q  3600000.0   6.25  B
   B   120.0  15000   0  4Q  1800000.0   0.00  C
   C   193.0  20000   1  4Q  3860000.0   0.52  B
대전 A   177.0  15000  16  4Q  2655000.0   9.04  B
   B   273.0  15000  11  4Q  4095000.0   4.03  A
   C   117.0  10000   3  4Q  1170000.0   2.56  C

In [ ]:
seoul = df2.loc['서울']
seoul

#loc을 사용해 서울만의 데이터를 불러왔기에 1개의 index가 있는 dataframe이 나온다.

,판매량,단가,반품,분기,매출,반품율,등급
상품,,,,,,,
A,179.0,15000,14,1Q,2685000.0,7.82,B
B,85.0,10000,4,1Q,850000.0,4.71,C
C,235.0,15000,12,1Q,3525000.0,5.11,B
A,222.0,10000,8,2Q,2220000.0,3.60,B
B,171.5,15000,3,2Q,2572500.0,1.75,B
C,164.0,20000,13,2Q,3280000.0,7.93,B
A,141.0,10000,16,3Q,1410000.0,11.35,C
B,171.5,20000,18,3Q,3430000.0,10.50,B
C,210.0,20000,10,3Q,4200000.0,4.76,A
